In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [4]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,2,176.044657
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,2,112.447373
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,2,139.176666
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,2,150.237064
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,2,54.380287
...,...,...,...,...,...,...,...,...,...,...,...
53995,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,9,0.000000
53996,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,9,0.000000
53997,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,9,0.000000
53998,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,9,0.000000


In [5]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [6]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    13500
mild          13500
moderate      13500
severe        13500
Name: sub_entity, dtype: int64

In [7]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.934756e+06
              2                  4.077938e+06
              3                  3.657270e+06
              4                  3.470352e+06
              5                  3.396960e+06
intervention  1                  4.934756e+06
              2                  4.077938e+06
              3                  3.657270e+06
              4                  3.470352e+06
              5                  3.396960e+06
zero          1                  4.934736e+06
              2                  4.077925e+06
              3                  3.657257e+06
              4                  3.470339e+06
              5                  3.396960e+06
Name: value, dtype: float64

In [8]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.638295e+06
              2                  2.075979e+06
              3                  1.766997e+06
              4                  1.565887e+06
              5                  1.319539e+06
intervention  1                  2.638295e+06
              2                  2.075979e+06
              3                  1.766997e+06
              4                  1.565887e+06
              5                  1.319539e+06
zero          1                  2.824121e+06
              2                  2.215088e+06
              3                  1.883333e+06
              4                  1.662475e+06
              5                  1.376956e+06
Name: value, dtype: float64

In [9]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.534635
              2                  0.509076
              3                  0.483147
              4                  0.451218
              5                  0.388447
intervention  1                  0.534635
              2                  0.509076
              3                  0.483147
              4                  0.451218
              5                  0.388447
zero          1                  0.572294
              2                  0.543190
              3                  0.514958
              4                  0.479053
              5                  0.405349
Name: value, dtype: float64

In [10]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [11]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,49649.700497
1,Female,0.0,0.019178,not_pregnant,2,43575.622144
2,Female,0.0,0.019178,not_pregnant,3,38689.356750
3,Female,0.0,0.019178,not_pregnant,4,36440.833935
4,Female,0.0,0.019178,not_pregnant,5,29202.585338
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,15176.692284
281,Male,95.0,125.000000,not_pregnant,2,15848.509818
282,Male,95.0,125.000000,not_pregnant,3,16370.176208
283,Male,95.0,125.000000,not_pregnant,4,17265.313670


In [12]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    5.312734e+06
2    4.479329e+06
3    4.040741e+06
4    3.799765e+06
5    3.668830e+06
Name: value, dtype: float64

In [13]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.840375e+06
              2                  2.280317e+06
              3                  1.952270e+06
              4                  1.714524e+06
              5                  1.425146e+06
intervention  1                  2.840375e+06
              2                  2.280317e+06
              3                  1.952270e+06
              4                  1.714524e+06
              5                  1.425146e+06
zero          1                  3.040447e+06
              2                  2.433126e+06
              3                  2.080811e+06
              4                  1.820288e+06
              5                  1.487158e+06
Name: value, dtype: float64

In [14]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,2,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,2,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,2,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,2,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,9,0.0
26996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,9,0.0
26997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,9,0.0
26998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,9,0.0


In [15]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [16]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.937808e+06
              2                  1.743659e+06
              3                  2.205352e+06
              4                  1.971860e+06
              5                  1.248927e+06
intervention  1                  2.937808e+06
              2                  1.743659e+06
              3                  2.205352e+06
              4                  1.971860e+06
              5                  1.248927e+06
zero          1                  3.017209e+06
              2                  1.790069e+06
              3                  2.254599e+06
              4                  2.008988e+06
              5                  1.264990e+06
Name: value, dtype: float64

In [17]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [18]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# TODO: Determine whether it's expected that the child_scenario column
# always contains only 'baseline'. If there's more than one scenario in
# this column, then results from different child scenarios would get
# added together in the call to aggregate_by_scenario below.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,7077.164512
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,5392.125343
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,5343.981366
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,5006.973532
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,5103.261485
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,866.591573
1196,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,722.159644
1197,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,625.871692
1198,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,385.151810


In [19]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  190842.721944
              2                  159404.705438
              3                  143902.345079
              4                  130807.183533
              5                  129796.160031
intervention  1                  190842.721944
              2                  159404.705438
              3                  143902.345079
              4                  130807.183533
              5                  129796.160031
zero          1                  191324.161707
              2                  159404.705438
              3                  144480.072794
              4                  130807.183533
              5                  129892.447983
Name: value, dtype: float64

In [20]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [21]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,40391.701768,zero
1,Female,0.0,0.019178,2,33290.554999,zero
2,Female,0.0,0.019178,3,29144.770544,zero
3,Female,0.0,0.019178,4,25661.946264,zero
4,Female,0.0,0.019178,5,19335.735573,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,7175.712146,intervention
746,Male,95.0,125.000000,2,6804.538012,intervention
747,Male,95.0,125.000000,3,6850.393434,intervention
748,Male,95.0,125.000000,4,6737.224336,intervention


In [22]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.201339e+08
              2                  1.125885e+08
              3                  1.112270e+08
              4                  1.060176e+08
              5                  1.004045e+08
intervention  1                  1.201339e+08
              2                  1.125885e+08
              3                  1.112270e+08
              4                  1.060176e+08
              5                  1.004045e+08
zero          1                  1.266749e+08
              2                  1.187605e+08
              3                  1.166266e+08
              4                  1.106754e+08
              5                  1.029664e+08
Name: value, dtype: float64

In [23]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.229743e+08
              2                  1.148689e+08
              3                  1.131792e+08
              4                  1.077321e+08
              5                  1.018296e+08
intervention  1                  1.229743e+08
              2                  1.148689e+08
              3                  1.131792e+08
              4                  1.077321e+08
              5                  1.018296e+08
zero          1                  1.297153e+08
              2                  1.211937e+08
              3                  1.187074e+08
              4                  1.124957e+08
              5                  1.044535e+08
Name: value, dtype: float64

In [24]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [25]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  5246.953936
              2                  4635.136215
              3                  4161.262947
              4                  3923.865884
              5                  3341.791859
baseline      1                  4876.988993
              2                  4356.327831
              3                  3949.653721
              4                  3750.364321
              5                  3275.558668
intervention  1                  2766.849667
              2                  2659.411546
              3                  2581.570609
              4                  2577.734970
              5                  2742.887884
Name: value, dtype: float64

In [26]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)